<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Lab pratique : analyse visuelle interactive avec Folium**

Temps estimé : **40** minutes

Le taux de succès d'un lancement peut dépendre de nombreux facteurs, comme la masse de la charge utile, le type d'orbite, etc. Il peut aussi dépendre de l'emplacement d'un site de lancement et de ses environs, c'est-à-dire de la position initiale des trajectoires des fusées. Trouver l'emplacement optimal pour construire un site de lancement fait certainement intervenir de nombreux facteurs, et nous pouvons espérer en découvrir quelques-uns en analysant les sites de lancement existants.

Dans les labs précédents d'analyse exploratoire des données, vous avez visualisé le jeu de données des lancements SpaceX avec `matplotlib` et `seaborn` et mis en évidence de premières corrélations entre le site de lancement et les taux de succès. Dans ce lab, vous allez réaliser une analyse visuelle plus interactive avec `Folium`.

## Objectifs

Ce lab comprend les tâches suivantes :

*   **TÂCHE 1 :** placer tous les sites de lancement sur une carte
*   **TÂCHE 2 :** marquer sur la carte les lancements réussis/échoués de chaque site
*   **TÂCHE 3 :** calculer les distances entre un site de lancement et ses environs

Une fois ces tâches terminées, vous devriez pouvoir identifier des tendances géographiques concernant les sites de lancement.

Commençons par importer les packages Python nécessaires à ce lab :

In [ ]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])

In [ ]:
import folium
import pandas as pd

In [ ]:
# Importer le plugin MarkerCluster de folium
from folium.plugins import MarkerCluster
# Importer le plugin MousePosition de folium
from folium.plugins import MousePosition
# Importer le plugin DivIcon de folium
from folium.features import DivIcon

Si vous avez besoin de revoir les bases de folium, vous pouvez télécharger et consulter le lab folium précédent :

[Générer des cartes avec Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)

## Tâche 1 : placer tous les sites de lancement sur une carte

Essayons d'abord d'ajouter l'emplacement de chaque site sur une carte à partir de ses coordonnées de latitude et de longitude.

Le jeu de données `spacex_launch_geo.csv` est une version enrichie dans laquelle la latitude et la longitude de chaque site ont été ajoutées.

In [ ]:
# Télécharger et lire le fichier `spacex_launch_geo.csv`
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df=pd.read_csv(spacex_csv_file)

Vous pouvez maintenant regarder les coordonnées de chaque site.

In [ ]:
# Sélectionner les sous-colonnes utiles : `Launch Site`, `Lat` (latitude), `Long` (longitude), `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

Les coordonnées ci-dessus ne sont que de simples nombres, qui ne donnent aucune idée intuitive de l'emplacement de ces sites de lancement. Si vous êtes très bon en géographie, vous pouvez les interpréter directement de tête ; sinon, ce n'est pas grave. Visualisons ces emplacements en les épinglant sur une carte.

Nous devons d'abord créer un objet `Map` de folium, centré au départ sur le centre spatial Johnson de la NASA à Houston, Texas.

In [ ]:
# Point de départ : centre spatial Johnson de la NASA
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

Nous pouvons utiliser `folium.Circle` pour ajouter une zone circulaire en surbrillance avec une étiquette de texte à une coordonnée donnée. Par exemple :

In [ ]:
# Créer un cercle au niveau du centre spatial Johnson de la NASA, avec une popup affichant son nom
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Créer un marqueur au même endroit, avec une icône qui affiche son nom
marker = folium.map.Marker(
    nasa_coordinate,
    # Créer une icône sous forme d'étiquette de texte
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

Vous devriez voir un petit cercle jaune près de la ville de Houston ; zoomez pour voir un cercle plus grand.

Ajoutons maintenant un cercle pour chaque site de lancement du dataframe `launch_sites_df`.

*À FAIRE :* créer et ajouter un `folium.Circle` et un `folium.Marker` pour chaque site de lancement sur la carte `site_map`.

Exemple de folium.Circle :

`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


Exemple de folium.Marker :

`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [ ]:
# Initialiser la carte
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Pour chaque site de lancement, ajouter un cercle selon ses coordonnées (Lat, Long)
# et le nom du site en popup, ainsi qu'un marqueur avec le nom en étiquette
for index, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    circle = folium.Circle(coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(row['Launch Site']))
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row['Launch Site'],
        )
    )
    site_map.add_child(circle)
    site_map.add_child(marker)

site_map

La carte générée avec les sites de lancement marqués devrait ressembler à ceci :

<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Vous pouvez maintenant explorer la carte en zoomant/dézoomant sur les zones marquées, et essayer de répondre aux questions suivantes :

*   Tous les sites de lancement sont-ils proches de la ligne de l'Équateur ?
*   Tous les sites de lancement sont-ils très proches de la côte ?

Essayez également d'expliquer vos conclusions.

> **Observation (à confirmer sur votre carte) :** les sites sont tous situés dans la moitié sud des États-Unis (Floride et Californie, vers 28° à 35° de latitude nord), donc plus près de l'Équateur que du nord du pays, mais pas « sur » l'Équateur. Ce choix s'explique par la vitesse de rotation de la Terre, plus élevée près de l'Équateur, qui donne un gain de vitesse gratuit au lancement. Tous les sites sont en revanche très proches de la côte : les trajectoires passent au-dessus de l'océan, ce qui limite les risques pour les zones habitées en cas de défaillance.

## Tâche 2 : marquer sur la carte les lancements réussis/échoués de chaque site

Améliorons maintenant la carte en ajoutant les issues des lancements de chaque site, afin de voir quels sites ont les taux de succès les plus élevés.
Rappelons que le dataframe spacex_df contient le détail des lancements, et que la colonne `class` indique si le lancement a réussi ou non.

In [ ]:
spacex_df.tail(10)

Créons ensuite des marqueurs pour tous les lancements.
Si un lancement a réussi `(class=1)`, nous utilisons un marqueur vert ; s'il a échoué `(class=0)`, un marqueur rouge.

Notez qu'un lancement n'a lieu que sur l'un des quatre sites de lancement : de nombreux lancements ont donc exactement les mêmes coordonnées. Les clusters de marqueurs (marker clusters) sont un bon moyen de simplifier une carte qui contient beaucoup de marqueurs aux mêmes coordonnées.

Créons d'abord un objet `MarkerCluster`.

In [ ]:
marker_cluster = MarkerCluster()


*À FAIRE :* créer dans le dataframe `spacex_df` une nouvelle colonne appelée `marker_color` qui stocke la couleur du marqueur en fonction de la valeur de `class`.

In [ ]:
# Appliquer une fonction qui teste la valeur de la colonne `class`
# Si class=1, marker_color vaut green
# Si class=0, marker_color vaut red
def assign_marker_color(launch_outcome):
    return 'green' if launch_outcome == 1 else 'red'

spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

*À FAIRE :* pour chaque lancement du dataframe `spacex_df`, ajouter un `folium.Marker` à `marker_cluster`.

In [ ]:
# Ajouter marker_cluster à la carte site_map actuelle
site_map.add_child(marker_cluster)

# Pour chaque ligne du dataframe spacex_df,
# créer un objet Marker avec ses coordonnées
# et personnaliser la propriété icon du marqueur pour indiquer si ce lancement a réussi ou échoué,
# par ex. icon=folium.Icon(color='white', icon_color=row['marker_color'])
for index, record in spacex_df.iterrows():
    marker = folium.Marker(
        [record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color'])
    )
    marker_cluster.add_child(marker)

site_map

Votre carte mise à jour peut ressembler aux captures d'écran suivantes :

<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


Grâce aux marqueurs colorés des clusters, vous devriez pouvoir identifier facilement les sites de lancement qui ont des taux de succès relativement élevés.

## Tâche 3 : calculer les distances entre un site de lancement et ses environs

Nous devons ensuite explorer et analyser les environs des sites de lancement.

Ajoutons d'abord un `MousePosition` sur la carte pour obtenir les coordonnées du point survolé par la souris. Ainsi, pendant que vous explorez la carte, vous pouvez facilement relever les coordonnées de n'importe quel point d'intérêt (comme une voie ferrée).

In [ ]:
# Ajouter MousePosition pour obtenir les coordonnées (Lat, Long) du point survolé par la souris
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Zoomez maintenant sur un site de lancement et explorez ses environs pour voir si vous trouvez facilement une voie ferrée, une autoroute, un littoral, etc. Placez la souris sur ces points et notez leurs coordonnées (affichées en haut à droite de la carte) afin de calculer la distance au site de lancement.

Zoomez maintenant sur un site de lancement et explorez ses environs pour voir si vous trouvez facilement une voie ferrée, une autoroute, un littoral, etc. Placez la souris sur ces points et notez leurs coordonnées (affichées en haut à droite de la carte) afin de calculer la distance au site de lancement.

In [ ]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # rayon approximatif de la Terre en km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*À FAIRE :* relever avec `MousePosition` un point sur le littoral le plus proche, puis calculer la distance entre ce point du littoral et le site de lancement.

In [ ]:
# Site de lancement étudié : CCAFS SLC-40 (coordonnées lues dans launch_sites_df)
site_row = launch_sites_df[launch_sites_df['Launch Site'] == 'CCAFS SLC-40'].iloc[0]
launch_site_lat, launch_site_lon = site_row['Lat'], site_row['Long']

# Coordonnées du point de littoral le plus proche (relevé avec MousePosition)
# ex. : Lat: 28.56367  Lon: -80.57163
coastline_lat, coastline_lon = 28.56367, -80.57163
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f"Distance au littoral : {distance_coastline:.2f} km")

In [ ]:
# Créer et ajouter un folium.Marker sur le point de littoral sélectionné
# Afficher la distance entre le littoral et le site de lancement via la propriété icon
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
        )
    )
site_map.add_child(distance_marker)

*À FAIRE :* tracer une `PolyLine` entre un site de lancement et le point de littoral sélectionné.

In [ ]:
# Créer un objet `folium.PolyLine` entre les coordonnées du littoral et celles du site de lancement
coordinates = [[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]]
lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

Votre carte mise à jour avec la ligne de distance devrait ressembler à la capture d'écran suivante :

<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*À FAIRE :* de la même façon, vous pouvez tracer une ligne entre un site de lancement et la ville, la voie ferrée, l'autoroute, etc. les plus proches. Vous devez d'abord utiliser `MousePosition` pour trouver leurs coordonnées sur la carte.

Un symbole de voie ferrée sur une carte peut ressembler à ceci :

<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


Un symbole d'autoroute sur une carte peut ressembler à ceci :

<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


Un symbole de ville sur une carte peut ressembler à ceci :

<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [ ]:
# Créer, pour la ville, la voie ferrée et l'autoroute les plus proches, un marqueur affichant la distance
# et tracer une ligne entre ce marqueur et le site de lancement.
# NB : coordonnées approximatives, à vérifier/affiner avec MousePosition sur votre carte.
proximities = {
    'Voie ferrée': (28.57205, -80.58527),
    'Autoroute':   (28.56342, -80.57674),
    'Ville (Titusville)': (28.6122, -80.8076),
}

distances = {'Littoral': distance_coastline}
for name, (lat, lon) in proximities.items():
    d = calculate_distance(launch_site_lat, launch_site_lon, lat, lon)
    distances[name] = d
    site_map.add_child(folium.Marker(
        [lat, lon],
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(d),
        )
    ))
    site_map.add_child(folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [lat, lon]], weight=1))

site_map

In [ ]:
# Récapitulatif des distances depuis CCAFS SLC-40
pd.DataFrame({'Point proche': list(distances.keys()),
              'Distance (km)': [round(v, 2) for v in distances.values()]})

Une fois les lignes de distance vers les points proches tracées, vous pouvez facilement répondre aux questions suivantes :

*   Les sites de lancement sont-ils proches des voies ferrées ?
*   Les sites de lancement sont-ils proches des autoroutes ?
*   Les sites de lancement sont-ils proches du littoral ?
*   Les sites de lancement restent-ils à une certaine distance des villes ?

Essayez également d'expliquer vos conclusions.

> **Observation (à confirmer sur votre carte) :** les sites sont très proches du littoral (de l'ordre du kilomètre), proches d'une voie ferrée et d'une route, ce qui facilite l'acheminement des équipements et du matériel lourd, mais loin des villes (plusieurs dizaines de kilomètres), ce qui limite les risques pour la population en cas d'accident.

# Prochaines étapes :

Vous avez maintenant découvert, de façon très interactive avec folium, plusieurs enseignements intéressants sur l'emplacement des sites de lancement. Vous allez ensuite construire un tableau de bord avec Plotly Dash à partir du détail des lancements.

## Auteurs

[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. Tous droits réservés. <h3/>